In [1]:
import numpy as np
import random
import csv

In [11]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)



def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

#  calcul a erorii cross-entropy
def cross_entropy_loss(y_true, y_predicted):
    return -np.sum(y_true * np.log(y_predicted + 1e-9)) / y_true.shape[0]


def initialize_population(population_size, num_weights):
    return [np.random.uniform(-1, 1, num_weights) for _ in range(population_size)]

# evaluare fitness pentru un individ
def fitness_eval(individ, X, y, structure):
    weights, biases = decode_individ(individ, structure)
    predictions = forward_pass(X, weights, biases)
    loss = cross_entropy_loss(y, predictions)
    return -loss  




# decodificare individ
def decode_individ(individ, structure):
    weights = []
    biases = []
    index = 0
    for i in range(len(structure) - 1):
        intrare_dim = structure[i]
        iesire_dim = structure[i + 1]
        weight_count = intrare_dim * iesire_dim
        bias_count = iesire_dim
        
        weights.append(individ[index:index + weight_count].reshape(intrare_dim, iesire_dim))
        index += weight_count
        biases.append(individ[index:index + bias_count])
        index += bias_count
    return weights, biases




# trecere prin retea
def forward_pass(X, weights, biases):
    layer_intrare = X
    
    for i in range(len(weights) - 1):
        layer_iesire = sigmoid(np.dot(layer_intrare, weights[i]) + biases[i])
        layer_intrare = layer_iesire
        
    final_output = softmax(np.dot(layer_intrare, weights[-1]) + biases[-1])
    return final_output






# selectia parintilorr pe baza fitnessului in turneu
def select_parents(population, fitness, tournament_size=3):
    parents = []
    for _ in range(2):
        tournament = random.sample(range(len(population)), tournament_size)
        best = max(tournament, key=lambda i: fitness[i])
        
        parents.append(population[best])
    return parents

# incrucisare intre 2 parinti
def crossover(parinte1, parinte2, crossover_rate=0.7):
    if random.random() < crossover_rate:
        point = random.randint(1, len(parinte1) - 1)
        
        copil1 = np.concatenate((parinte1[:point], parinte2[point:]))
        
        copil2 = np.concatenate((parinte2[:point], parinte1[point:]))
        return copil1, copil2
    return parinte1.copy(), parinte2.copy()




# mutarea unui individ
def mutate(individual, mutation_rate=0.1):
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            
            individual[i] += np.random.normal(0, 0.1)
    return individual

def train_evolutionary(X, y, structure, population_size=50, generations=50, mutation_rate=0.2):
    nr_weights = sum(structure[i] * structure[i + 1] + structure[i + 1] for i in range(len(structure) - 1))
    
    population = initialize_population(population_size, nr_weights)
    
    for generation in range(generations):
        fitness = [fitness_eval(individ, X, y, structure) for individ in population]
        new_population = []

        
        for _ in range(population_size // 2):
            parinte1, parinte2 = select_parents(population, fitness)
            copil1, copil2 = crossover(parinte1, parinte2)
            
            new_population.append(mutate(copil1, mutation_rate))
            new_population.append(mutate(copil2, mutation_rate))

        population = new_population
        best_fitness = max(fitness)
        
        print(f"generatia {generation + 1}: cel mai bun fitness = {best_fitness}")

    best_individ = population[np.argmax(fitness)]
    weights, biases = decode_individ(best_individ, structure)
    return weights, biases



def evaluate_performance(predictions, labels):
    predicted_classes = np.argmax(predictions, axis=1) + 1
    correct = np.sum(predicted_classes == labels)
    
    accuracy = correct / len(labels)
    print(f"acuratetea retelei: {accuracy * 100:.2f}%")
    return accuracy

In [12]:
def load_data(filename):
    data = []
    labels = []
    
    
    with open(filename, 'r') as file:
        reader = csv.reader(file, delimiter=';')
        next(reader) 
        for row in reader:
            data.append(list(map(float, row[:-1])))
            labels.append(int(row[-1]))
            
    return np.array(data), np.array(labels)

In [13]:
data, wine_quality_labels=load_data("winequality-red.csv");
print(data[0:4])
print(len(data[0]))
print(len(wine_quality_labels))
print(wine_quality_labels[0:4])

[[7.400e+00 7.000e-01 0.000e+00 1.900e+00 7.600e-02 1.100e+01 3.400e+01
  9.978e-01 3.510e+00 5.600e-01 9.400e+00]
 [7.800e+00 8.800e-01 0.000e+00 2.600e+00 9.800e-02 2.500e+01 6.700e+01
  9.968e-01 3.200e+00 6.800e-01 9.800e+00]
 [7.800e+00 7.600e-01 4.000e-02 2.300e+00 9.200e-02 1.500e+01 5.400e+01
  9.970e-01 3.260e+00 6.500e-01 9.800e+00]
 [1.120e+01 2.800e-01 5.600e-01 1.900e+00 7.500e-02 1.700e+01 6.000e+01
  9.980e-01 3.160e+00 5.800e-01 9.800e+00]]
11
1599
[5 5 5 6]


In [19]:
    nr_clas = 10
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,6,7,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure)

    
    
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, wine_quality_labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.9152529542801235
generatia 2: cel mai bun fitness = -1.9119041415022096
generatia 3: cel mai bun fitness = -1.7514952551140706
generatia 4: cel mai bun fitness = -1.579138348718241
generatia 5: cel mai bun fitness = -1.670942570122676
generatia 6: cel mai bun fitness = -1.5603060735561545
generatia 7: cel mai bun fitness = -1.3942371798373419
generatia 8: cel mai bun fitness = -1.4050360544880796
generatia 9: cel mai bun fitness = -1.3993752855314516
generatia 10: cel mai bun fitness = -1.385388621435809
generatia 11: cel mai bun fitness = -1.351014790944881
generatia 12: cel mai bun fitness = -1.3314248833363693
generatia 13: cel mai bun fitness = -1.3213461248279794
generatia 14: cel mai bun fitness = -1.2888294965827973
generatia 15: cel mai bun fitness = -1.2847257454097094
generatia 16: cel

0.49280800500312694

In [15]:
print(len(predictions))

1599


In [24]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,3,4,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=100,generations=30, mutation_rate=0.7)

    
    
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, wine_quality_labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.7782775273523426
generatia 2: cel mai bun fitness = -1.561685136505267
generatia 3: cel mai bun fitness = -1.5314793030944
generatia 4: cel mai bun fitness = -1.5003587580140467
generatia 5: cel mai bun fitness = -1.4560568428608252
generatia 6: cel mai bun fitness = -1.4201375966094179
generatia 7: cel mai bun fitness = -1.4008620548131618
generatia 8: cel mai bun fitness = -1.3860310589167821
generatia 9: cel mai bun fitness = -1.357366753226231
generatia 10: cel mai bun fitness = -1.3198210062130047
generatia 11: cel mai bun fitness = -1.3027129652344258
generatia 12: cel mai bun fitness = -1.274627883742821
generatia 13: cel mai bun fitness = -1.2560286511830165
generatia 14: cel mai bun fitness = -1.2509302506467046
generatia 15: cel mai bun fitness = -1.2449083856486587
generatia 16: cel m

0.4427767354596623

In [21]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    # structura retelei
    structure = [11,7,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=75,generations=50, mutation_rate=0.4)

    
    
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, wine_quality_labels)
    

generatia 1: cel mai bun fitness = -1.6431164698094884
generatia 2: cel mai bun fitness = -1.5928692244686213
generatia 3: cel mai bun fitness = -1.5607818838741239
generatia 4: cel mai bun fitness = -1.4304609363538592
generatia 5: cel mai bun fitness = -1.4250843715137278
generatia 6: cel mai bun fitness = -1.4311356855060713
generatia 7: cel mai bun fitness = -1.3747410719620499
generatia 8: cel mai bun fitness = -1.3501314767126007
generatia 9: cel mai bun fitness = -1.3069363187144014
generatia 10: cel mai bun fitness = -1.3232316600039846
generatia 11: cel mai bun fitness = -1.310914681254772
generatia 12: cel mai bun fitness = -1.2838335154269538
generatia 13: cel mai bun fitness = -1.261839010284606
generatia 14: cel mai bun fitness = -1.2614531310311767
generatia 15: cel mai bun fitness = -1.2590194563025605
generatia 16: cel mai bun fitness = -1.2553134431400153
generatia 17: cel mai bun fitness = -1.2200183852241704
generatia 18: cel mai bun fitness = -1.211727330063725
gene

0.43214509068167606

In [23]:
    y = np.zeros((len(wine_quality_labels), nr_clas))
    for i, label in enumerate(wine_quality_labels):
        y[i, label - 1] = 1
    print(y)
    # structura retelei
    structure = [11,5,5,10]

    # antrenare
    weights, biases = train_evolutionary(data, y, structure,population_size=100,generations=300, mutation_rate=0.5)

    
    
    predictions = forward_pass(data, weights, biases)
    print("predictii obtinute:", np.argmax(predictions, axis=1) + 1)
    evaluate_performance(predictions, wine_quality_labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
generatia 1: cel mai bun fitness = -1.6415084957686006
generatia 2: cel mai bun fitness = -1.6130269226721998
generatia 3: cel mai bun fitness = -1.417598526847827
generatia 4: cel mai bun fitness = -1.3530668133661412
generatia 5: cel mai bun fitness = -1.313167435493837
generatia 6: cel mai bun fitness = -1.304218496968567
generatia 7: cel mai bun fitness = -1.2907327028933673
generatia 8: cel mai bun fitness = -1.2733200056004994
generatia 9: cel mai bun fitness = -1.2572331012369031
generatia 10: cel mai bun fitness = -1.2468011053453314
generatia 11: cel mai bun fitness = -1.239268672561522
generatia 12: cel mai bun fitness = -1.2356847204698118
generatia 13: cel mai bun fitness = -1.222363771551851
generatia 14: cel mai bun fitness = -1.2169568944419729
generatia 15: cel mai bun fitness = -1.2086263997333788
generatia 16: cel 

generatia 146: cel mai bun fitness = -1.1324809768608506
generatia 147: cel mai bun fitness = -1.1295139948624313
generatia 148: cel mai bun fitness = -1.132139219619905
generatia 149: cel mai bun fitness = -1.1297715381555655
generatia 150: cel mai bun fitness = -1.1324568033617441
generatia 151: cel mai bun fitness = -1.128545764976315
generatia 152: cel mai bun fitness = -1.133752755990643
generatia 153: cel mai bun fitness = -1.1319379429753267
generatia 154: cel mai bun fitness = -1.1349243058259364
generatia 155: cel mai bun fitness = -1.131722233567066
generatia 156: cel mai bun fitness = -1.128064401795332
generatia 157: cel mai bun fitness = -1.132192662128623
generatia 158: cel mai bun fitness = -1.1315197879475682
generatia 159: cel mai bun fitness = -1.1273159545974734
generatia 160: cel mai bun fitness = -1.1296230504472562
generatia 161: cel mai bun fitness = -1.1274663387104618
generatia 162: cel mai bun fitness = -1.1309384202543815
generatia 163: cel mai bun fitness = 

generatia 294: cel mai bun fitness = -1.1258014699126468
generatia 295: cel mai bun fitness = -1.124721346882183
generatia 296: cel mai bun fitness = -1.1234793936061502
generatia 297: cel mai bun fitness = -1.121037263181651
generatia 298: cel mai bun fitness = -1.1243465782760658
generatia 299: cel mai bun fitness = -1.1238813655580289
generatia 300: cel mai bun fitness = -1.1252292681391078
predictii obtinute: [5 5 5 ... 6 6 5]
acuratetea retelei: 50.53%


0.5053158223889931